In [8]:
import os
import tempfile
import subprocess
import numpy as np
import matplotlib.pyplot as plt

In [9]:
def save_force_video(keypoints, forces, out_path, fps=30, scale=5, figsize=(6,6), dpi=100):
    keypoints = np.asarray(keypoints)
    forces = np.asarray(forces)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    x_min = np.nanmin(keypoints[:,:,0]) - 20
    x_max = np.nanmax(keypoints[:,:,0]) + 20
    y_min = np.nanmin(keypoints[:,:,1]) - 20
    y_max = np.nanmax(keypoints[:,:,1]) + 20

    skeleton = [
        (5,7),(7,9),
        (6,8),(8,10),
        (5,6),
        (5,11),(6,12),
        (11,12),
        (11,13),(13,15),
        (12,14),(14,16)
    ]

    with tempfile.TemporaryDirectory() as tmpdir:
        for frame_idx in range(keypoints.shape[0]):
            k = keypoints[frame_idx]
            f = forces[frame_idx]

            fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
            ax.scatter(k[:,0], k[:,1], c='red')
            for i, j in skeleton:
                ax.plot([k[i,0], k[j,0]], [k[i,1], k[j,1]], 'blue')

            if not np.any(np.isnan(k[16])):
                x, y = k[16]
                fx, fy, fz = f[0], f[1], f[2]
                ax.arrow(x, y, fx * scale, -fy * scale, color='purple', head_width=5)

            if not np.any(np.isnan(k[15])):
                x, y = k[15]
                fx, fy, fz = f[3], f[4], f[5]
                ax.arrow(x, y, fx * scale, -fy * scale, color='purple', head_width=5)

            ax.set_title(f'Frame {frame_idx}')
            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_max, y_min)
            ax.set_aspect('equal')

            filename = os.path.join(tmpdir, f'frame_{frame_idx:04d}.png')
            fig.savefig(filename, dpi=dpi)
            plt.close(fig)

        cmd = [
            'ffmpeg',
            '-y',
            '-framerate', str(fps),
            '-i', os.path.join(tmpdir, 'frame_%04d.png'),
            '-c:v', 'libx264',
            '-pix_fmt', 'yuv420p',
            out_path,
        ]
        print('Running ffmpeg...')
        subprocess.run(cmd, check=True)

    print(f'Video saved to {out_path}')

In [13]:
kps = np.load("out_files/skeleton_Subject7_Counter_Movement_Jump01.npy")

forces = np.load("out_files/gt_Subject7_Counter_Movement_Jump01.npy")
# forces = pred.squeeze(axis=1)

In [14]:
kps.shape, forces.shape

((231, 17, 2), (231, 6))

In [15]:
output_video = 'out_files/gt_force_vis.mp4'
save_force_video(kps, forces, out_path=output_video, fps=30, scale=8)
print('Saved video to', output_video)

Running ffmpeg...
Video saved to out_files/gt_force_vis.mp4
Saved video to out_files/gt_force_vis.mp4


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena